# Lenta — STEP 2: pre-label sliced frames on Google Colab

**Use this when your laptop has no GPU.** It runs *our* detector
(the OpenFoodFacts price-tag YOLO — the same weights the original
auto-label notebook used) over the frames you sliced in STEP 1, and
writes a YOLO `.txt` next to every image. You then bring the folder
back to the laptop for STEP 3.

### What to press

1. **Runtime → Change runtime type → Hardware accelerator: T4 GPU → Save.**
2. On your laptop run STEP 1 (`slice_video_frames.py`). It made a folder
   `cvat_video_frames/` with one subfolder per video full of JPEGs.
3. Upload that whole `cvat_video_frames/` folder to **Google Drive** at
   `MyDrive/lenta_prelabel/cvat_video_frames` (drag-and-drop in
   drive.google.com — a few minutes).
4. Run the cells below **top to bottom** (▶ on each, or *Runtime → Run all*).
5. When done, the **same Drive folder** now also has a `.txt` next to
   every `.jpg`. Download it back to the laptop (replace the local
   `cvat_video_frames/`) and continue with STEP 3.

No code editing needed unless you want a different model (CELL 2).

In [ ]:
# CELL 0 — mount Google Drive and point at the uploaded frames folder
from google.colab import drive
drive.mount("/content/drive")

# Where you uploaded STEP 1's output. Change ONLY if you used another path.
FRAMES_DIR = "/content/drive/MyDrive/lenta_prelabel/cvat_video_frames"

import os
assert os.path.isdir(FRAMES_DIR), (
    f"Not found: {FRAMES_DIR}\n"
    "Upload STEP 1's cvat_video_frames/ folder there first."
)
print("frames dir OK:", FRAMES_DIR)
print("scenes:", sorted(d for d in os.listdir(FRAMES_DIR)
                          if os.path.isdir(os.path.join(FRAMES_DIR, d))))

In [ ]:
# CELL 1 — install the detector runtime (~1 min)
!pip -q install ultralytics huggingface_hub hf_xet

In [ ]:
# CELL 2 — download our detector weights
# Default = the OpenFoodFacts price-tag detector (the fixed base of our
# solution, exactly what the original auto-label notebook used).
# To use a fine-tuned checkpoint instead: upload it to Drive and set
#   MODEL_PATH = "/content/drive/MyDrive/.../best.pt"
# then skip the hf_hub_download lines.
from huggingface_hub import hf_hub_download

MODEL_PATH = hf_hub_download(
    repo_id="openfoodfacts/price-tag-detection",
    filename="weights/best.pt",
)
print("model:", MODEL_PATH)

In [ ]:
# CELL 3 — run the detector over every frame, write YOLO .txt next to it
# Recall-first defaults (conf 0.05 / iou 0.50 / imgsz 1280): better to
# delete a wrong box in CVAT than to hand-draw a missed one. An empty
# .txt means "checked, no tag" — a useful negative; keep it.
from pathlib import Path
from ultralytics import YOLO

CONF, IOU, IMGSZ = 0.05, 0.50, 1280
IMG_EXTS = (".jpg", ".jpeg", ".png", ".bmp")

net = YOLO(MODEL_PATH)
root = Path(FRAMES_DIR)
scene_dirs = [d for d in sorted(root.iterdir()) if d.is_dir()] or [root]

grand_img = grand_box = 0
for sdir in scene_dirs:
    imgs = sorted(p for p in sdir.iterdir()
                  if p.is_file() and p.suffix.lower() in IMG_EXTS)
    if not imgs:
        continue
    n_box = boxed = 0
    results = net.predict(source=[str(p) for p in imgs], stream=True,
                          verbose=False, conf=CONF, iou=IOU, imgsz=IMGSZ)
    for img_path, res in zip(imgs, results):
        lines = []
        b = getattr(res, "boxes", None)
        if b is not None and len(b):
            for cx, cy, bw, bh in (t.tolist() for t in b.xywhn):
                if bw > 0 and bh > 0:
                    lines.append(f"0 {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}")
        img_path.with_suffix(".txt").write_text(
            "\n".join(lines) + ("\n" if lines else ""), encoding="utf-8")
        n_box += len(lines)
        boxed += 1 if lines else 0
    grand_img += len(imgs)
    grand_box += n_box
    print(f"{sdir.name}: {len(imgs)} imgs, {n_box} boxes ({boxed} boxed)")

print(f"\nDONE: {grand_img} frames, {grand_box} pre-labelled boxes.")
print("The .txt files are saved in-place on Drive — they sync automatically.")

In [ ]:
# CELL 4 (optional) — also make a single zip to download instead of
# waiting for Drive sync. Skip if you'll just re-download the Drive folder.
import shutil
from google.colab import files

zip_base = "/content/cvat_video_frames_prelabelled"
shutil.make_archive(zip_base, "zip", FRAMES_DIR)
files.download(zip_base + ".zip")

## Back on the laptop (STEP 3)

Make sure the local `cvat_video_frames/` folder now has a `.txt` next to
every `.jpg` (download the Drive folder or unzip CELL 4's archive over it),
then:

```bash
.venv/Scripts/python.exe projects/price_tag_pipeline/scripts/frames_to_cvat.py \
  --frames-dir cvat_video_frames --out cvat_seeds_video
.venv/Scripts/python.exe projects/price_tag_pipeline/scripts/cvat_bootstrap_photos.py \
  --manifest cvat_seeds_video/manifest.json --user admin --password ***
```

Then open http://localhost:8080, validate/fix the boxes, export, and
convert — see `docs/runbooks/annotation-cvat.md`.